# **Monitoramento Automatizado de Obras via Visão Computacional**

**Projeto Acadêmico:** Classificação de Fases de Construção com YOLOv8

**Aluno:** Carlos Frederico Coelho Lisboa

**Fases Monitoradas:** Fundação, Alvenaria, Cobertura e Revestimento Externo.

---
Este notebook apenas **orquestra** o fluxo de trabalho. Toda a lógica fica no pacote `obra_classifier` (pasta `src/`) do repositório [FredLisboa77/Acompanhamento-de-Obra](https://github.com/FredLisboa77/Acompanhamento-de-Obra), onde é versionada e testada.

**No Colab:** selecione *Ambiente de execução → Alterar o tipo → GPU T4* e execute as células em ordem.

**0. Preparação do ambiente**

Clona o repositório (no Colab) e instala o pacote com as dependências de treino e da interface. Fora do Colab, basta abrir o notebook a partir do repositório já instalado (`pip install -e ".[train,app]"`).

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/FredLisboa77/Acompanhamento-de-Obra.git"
REPO_BRANCH = "main"
EM_COLAB = "google.colab" in sys.modules

if EM_COLAB:
    RAIZ = Path("/content/Acompanhamento-de-Obra")
    if not RAIZ.exists():
        !git clone -q --branch {REPO_BRANCH} {REPO_URL} {RAIZ}
    %pip install -q -e "{RAIZ}[train,app]"
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(RAIZ)
# Instalações editáveis só entram no sys.path após reiniciar o kernel; isto evita o reinício.
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))
print(f"Diretório do projeto: {RAIZ}")

**1. Módulo de Gestão de Dados e Curadoria**

Este módulo é destinado à preparação do ambiente e garantia de que os dados brutos estão íntegros antes de qualquer processamento.

**1.1. Conexão e Setup**

Montamos o Google Drive para acessar o dataset e definimos os caminhos que guiarão o fluxo de trabalho. O dataset bruto deve ter **uma subpasta por classe**.

In [ ]:
from obra_classifier.config import CLASSES, ConfigDataset, ConfigTreino

if EM_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PASTA_DRIVE = Path("/content/drive/MyDrive/Trabalho - Visão Computacional")
else:
    PASTA_DRIVE = RAIZ / "data"

PATH_DATASET_BRUTO = PASTA_DRIVE / "Dataset_Bruto"
PATH_DATASET_PROCESSADO = RAIZ / "data" / "processado"
PATH_MODELO = RAIZ / "models" / "best.pt"
PATH_MODELO_BACKUP = PASTA_DRIVE / "best.pt"  # cópia persistente entre sessões

CONFIG_DATASET = ConfigDataset()
CONFIG_TREINO = ConfigTreino()

if not PATH_DATASET_BRUTO.is_dir():
    raise FileNotFoundError(
        f"Dataset não encontrado em {PATH_DATASET_BRUTO}. Verifique o nome da pasta."
    )
print(f"✅ Dataset encontrado: {PATH_DATASET_BRUTO}")

**1.2. Inspeção de Dados**

Antes de treinar, precisamos entender a distribuição das classes e validar visualmente se as imagens estão corretamente rotuladas nas subpastas. Classes muito desbalanceadas tendem a enviesar o modelo para a classe majoritária.

In [ ]:
import matplotlib.pyplot as plt

from obra_classifier.dataset import contar_por_classe
from obra_classifier.visualizacao import exibir_amostras

contagem = contar_por_classe(PATH_DATASET_BRUTO, CLASSES)
for classe, total in contagem.items():
    print(f"{classe:<22} {total:>4} imagens")

if sum(contagem.values()) == 0:
    raise RuntimeError("Nenhuma imagem encontrada. Verifique PATH_DATASET_BRUTO.")

exibir_amostras(PATH_DATASET_BRUTO, CLASSES)
plt.show()

**1.3. Divisão Treino/Validação e Data Augmentation (Albumentations)**

Para aumentar a robustez do modelo frente a variações climáticas e de iluminação comuns em canteiros de obras, aplicamos transformações sintéticas (brilho/contraste, gama, espelhamento, desfoque e CLAHE).

⚠️ **A divisão treino/validação é feita antes do augmentation**, e as variações são geradas **somente no treino**. Se as cópias aumentadas de uma mesma foto aparecessem nos dois conjuntos, o modelo seria avaliado em imagens que ele praticamente já viu (*data leakage*), e a acurácia de validação ficaria artificialmente próxima de 100%.

Imagens corrompidas são detectadas e ignoradas. A semente fixa torna o resultado reprodutível.

In [ ]:
from obra_classifier.augmentation import criar_pipeline
from obra_classifier.dataset import preparar_dataset

resumo = preparar_dataset(
    PATH_DATASET_BRUTO,
    PATH_DATASET_PROCESSADO,
    criar_pipeline(CONFIG_DATASET.seed),
    CLASSES,
    CONFIG_DATASET,
)

for classe in CLASSES:
    print(
        f"{classe:<22} treino={resumo.treino[classe]:>4} "
        f"(+{resumo.aumentadas[classe]} aumentadas)  val={resumo.validacao[classe]:>4}"
    )
if resumo.ignoradas:
    print(f"⚠️ {len(resumo.ignoradas)} imagem(ns) ilegível(is) ignorada(s):")
    for caminho in resumo.ignoradas:
        print(f"   - {caminho}")

**1.4. Treinamento do Modelo**

⚠️ O treinamento requer GPU ativa e tempo considerável de processamento. **Se o modelo já foi treinado, pule esta célula**: o módulo 2 carrega o `best.pt` salvo no Drive.

Partimos do YOLOv8 Nano pré-treinado no ImageNet (*transfer learning*). O *early stopping* (`paciencia`) interrompe o treino quando a validação para de melhorar. Ao final, o `best.pt` é copiado para `models/` (de onde o app o lê) e para o Drive.

In [ ]:
from obra_classifier.treino import exportar_modelo, treinar

melhor = treinar(PATH_DATASET_PROCESSADO, CONFIG_TREINO)
exportar_modelo(melhor, PATH_MODELO)
if EM_COLAB:
    exportar_modelo(melhor, PATH_MODELO_BACKUP)
print(f"✅ Modelo salvo em {PATH_MODELO}")

**2. Módulo de Avaliação**

Este módulo permite testar a eficácia do modelo treinado.

**2.1. Carregamento do Modelo**

Procuramos os pesos treinados (`best.pt`) nesta ordem: pasta `models/` (treino desta sessão), cópia no Google Drive e, por fim, um link público do Drive em `LINK_DO_DRIVE` (download via `gdown`).

In [ ]:
import shutil

from obra_classifier.predicao import carregar_modelo

LINK_DO_DRIVE = ""  # opcional: link público do best.pt

if not PATH_MODELO.is_file():
    PATH_MODELO.parent.mkdir(parents=True, exist_ok=True)
    if PATH_MODELO_BACKUP.is_file():
        shutil.copy2(PATH_MODELO_BACKUP, PATH_MODELO)
    elif LINK_DO_DRIVE:
        import gdown

        gdown.download(LINK_DO_DRIVE, str(PATH_MODELO), fuzzy=True)

modelo = carregar_modelo(PATH_MODELO)
print(f"✅ Modelo carregado: {PATH_MODELO} | classes: {list(modelo.names.values())}")

**2.2. Métricas na Validação**

A acurácia *top-1* é medida apenas em imagens originais que o modelo **não viu** no treino. A matriz de confusão mostra quais fases são confundidas entre si.

In [ ]:
from IPython.display import Image as ImagemIPython
from IPython.display import display

from obra_classifier.treino import avaliar

metricas = avaliar(PATH_MODELO, PATH_DATASET_PROCESSADO, CONFIG_TREINO.tamanho_imagem)
print(f"Acurácia top-1: {metricas['top1']:.1%} | top-5: {metricas['top5']:.1%}")

matriz = CONFIG_TREINO.projeto.resolve() / CONFIG_TREINO.nome / "confusion_matrix_normalized.png"
if matriz.is_file():
    display(ImagemIPython(filename=str(matriz), width=500))

**2.3. Interface Streamlit e Túnel de Acesso**

A interface está versionada em `app/streamlit_app.py`. A célula abaixo a inicia em segundo plano e a expõe na internet por um túnel temporário da Cloudflare (`cloudflared`, sem conta nem token).

*Instruções:*

a - Execute a célula abaixo.

b - Clique no link terminado em `.trycloudflare.com`.

c - Envie uma foto da obra (ou escolha um exemplo) e veja a fase identificada.

O link é público enquanto a sessão estiver ativa. Se o upload de imagens falhar com erro 403, altere `DESATIVAR_XSRF` para `True` e execute de novo.

In [ ]:
import re
import subprocess
import time
import urllib.request

PORTA = 8501
DESATIVAR_XSRF = False
TIMEOUT_S = 60
CLOUDFLARED = RAIZ / "cloudflared"
URL_CLOUDFLARED = (
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
)


def aguardar_streamlit(porta: int, timeout_s: int) -> None:
    """Espera o endpoint de saúde do Streamlit responder, em vez de um sleep fixo."""
    limite = time.monotonic() + timeout_s
    while time.monotonic() < limite:
        try:
            with urllib.request.urlopen(f"http://localhost:{porta}/_stcore/health", timeout=2):
                return
        except OSError:
            time.sleep(1)
    raise TimeoutError("O Streamlit não respondeu a tempo.")


streamlit_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "app/streamlit_app.py",
        "--server.port",
        str(PORTA),
        "--server.headless",
        "true",
        "--server.enableXsrfProtection",
        str(not DESATIVAR_XSRF).lower(),
    ],
    env={**os.environ, "OBRA_MODELO": str(PATH_MODELO)},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
aguardar_streamlit(PORTA, TIMEOUT_S)
print("✅ Streamlit iniciado.")

if not CLOUDFLARED.exists():
    urllib.request.urlretrieve(URL_CLOUDFLARED, CLOUDFLARED)
    CLOUDFLARED.chmod(0o755)

tunel = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--url", f"http://localhost:{PORTA}"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
)

print("Aguardando URL do túnel...")
for linha in tunel.stderr:
    if encontrado := re.search(r"https://[\w-]+\.trycloudflare\.com", linha):
        print(f"🌐 Interface disponível em: {encontrado.group(0)}")
        break
else:
    raise RuntimeError("O cloudflared encerrou sem fornecer uma URL.")

**3. Limitações do Modelo**

- **Dataset pequeno:** algumas dezenas de fotos por classe limitam a capacidade de generalização; o augmentation ajuda, mas não substitui imagens reais de obras distintas.
- **Fronteiras entre fases:** a transição entre alvenaria e revestimento externo (ou entre fundação e início da alvenaria) é gradual, e uma única foto pode conter as duas fases. Um classificador de rótulo único precisa escolher uma delas.
- **Oclusão:** tapumes, andaimes, telas de proteção e veículos podem esconder justamente os elementos que caracterizam a fase.
- **Iluminação e clima:** dias nublados, contraluz e sombras fortes alteram cores e texturas; o augmentation simula parte dessas variações.
- **Ângulo e distância:** fotos muito próximas ou aéreas diferem das imagens de treino.
- **Uso em decisões de crédito:** a saída deve ser tratada como **apoio** à vistoria, acompanhada da confiança da predição, e não como laudo definitivo.